# Non-Equilibrium Financial Phase-Transition Detection and Irreversibility-Aware Crisis Resolution Using Lightweight Causal Digital Twins

This Jupyter Notebook implements an end-to-end 5-stage framework for detecting non-equilibrium financial phase transitions, modeling causal stress propagation, running counterfactual digital twin simulations, and optimizing crisis interventions with irreversibility constraints.

---
### Framework Stages Overview:
- **Stage 1**: Financial Data Integration & Non-Equilibrium State Construction
- **Stage 2**: Non-Equilibrium Transition Fingerprint (NETF) for Phase-Transition Discovery
- **Stage 3**: Irreversibility-Constrained Causal Twin (ICCT) for Stress Propagation & Counterfactual Simulation
- **Stage 4**: Reversibility-Budgeted Crisis Controller (RBCC) for Decision-Making
- **Stage 5**: Integrated Validation, Robustness Analysis, & Computational Profiling


In [ ]:
import time
import psutil
import os
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import Lasso, Ridge, LogisticRegression
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import roc_auc_score, precision_recall_fscore_support

# Set seed for reproducibility
np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("All core dependencies imported successfully.")


## Stage 1 — Financial Data Integration & Non-Equilibrium State Construction


In [ ]:
# Synthetic Generation of Multi-Source Macro-Financial Panel Data
dates = pd.date_range(start='2000-01-01', end='2023-12-01', freq='MS')
N = len(dates)

# Generating synthetic macro-financial indicators mimicking FRED, BIS, ECB, and NY Fed
data = {
    'Date': dates,
    'FRED_Yield_Spread': np.sin(np.linspace(0, 10, N)) + np.random.normal(0, 0.2, N),
    'FRED_Credit_Cond': np.cos(np.linspace(0, 8, N)) + np.random.normal(0, 0.15, N),
    'BIS_Credit_to_GDP': np.linspace(50, 80, N) + np.random.normal(0, 0.5, N),
    'ECB_Bank_BalanceSheet': np.random.normal(100, 5, N),
    'NYFed_Financial_Stress': np.random.exponential(0.5, N)
}

df_raw = pd.DataFrame(data)

# Inject synthetic crisis events (e.g. 2008 GFC and 2020 Shock)
df_raw.loc[(df_raw['Date'] >= '2007-07-01') & (df_raw['Date'] <= '2009-06-01'), 'NYFed_Financial_Stress'] += 3.5
df_raw.loc[(df_raw['Date'] >= '2007-07-01') & (df_raw['Date'] <= '2009-06-01'), 'FRED_Credit_Cond'] -= 2.0

df_raw.loc[(df_raw['Date'] >= '2020-03-01') & (df_raw['Date'] <= '2020-12-01'), 'NYFed_Financial_Stress'] += 4.0

# Point-in-time alignment (Simulating publication lags)
df_aligned = df_raw.copy()
df_aligned['FRED_Yield_Spread'] = df_aligned['FRED_Yield_Spread'].shift(1) # 1 month lag
df_aligned['BIS_Credit_to_GDP'] = df_aligned['BIS_Credit_to_GDP'].shift(3) # 1 quarter lag
df_aligned.bfill(inplace=True)

# Feature Scaling via Robust Median-IQR Scaler
feature_cols = ['FRED_Yield_Spread', 'FRED_Credit_Cond', 'BIS_Credit_to_GDP', 'ECB_Bank_BalanceSheet', 'NYFed_Financial_Stress']
scaler = RobustScaler()
scaled_features = scaler.fit_transform(df_aligned[feature_cols])
df_scaled = pd.DataFrame(scaled_features, columns=[f"{col}_scaled" for col in feature_cols])
df_state = pd.concat([df_aligned[['Date']], df_scaled], axis=1)

# Regime Labelling Rules
def assign_labels(row):
    stress = row['NYFed_Financial_Stress_scaled']
    if stress > 1.8:
        return 2 # Crisis
    elif 0.8 < stress <= 1.8:
        return 1 # Transition
    else:
        return 0 # Stable

df_state['phase_state'] = df_state.apply(assign_labels, axis=1)
df_state['stress_level'] = df_state['NYFed_Financial_Stress_scaled'].clip(lower=0)
df_state['transition_within_h'] = (df_state['phase_state'].shift(-3) == 2).astype(int)

print(f"Dataset constructed. Total observations: {len(df_state)}")
print(df_state['phase_state'].value_counts().rename({0: 'Stable', 1: 'Transition', 2: 'Crisis'}))


## Stage 2 — Non-Equilibrium Transition Fingerprint (NETF)


In [ ]:
# Calculate Critical Slowing Down (CSD) indicators (Rolling AR(1) & Fluctuation Variance)
window = 12

def rolling_ar1(val):
    if len(val) < 3: return 0
    return np.corrcoef(val[:-1], val[1:])[0, 1]

df_netf = df_state.copy()
target_col = 'NYFed_Financial_Stress_scaled'

# 1. Critical slowing down statistics
df_netf['ar1_autocorr'] = df_netf[target_col].rolling(window).apply(rolling_ar1, raw=True).fillna(0)
df_netf['rolling_variance'] = df_netf[target_col].rolling(window).var().fillna(0)

# 2. Windowed Fluctuation-Dependence
scaled_cols = [c for c in df_state.columns if c.endswith('_scaled')]
df_netf['cross_correlation_mean'] = df_netf[scaled_cols].rolling(window).corr().groupby(level=0).mean().mean(axis=1).fillna(0)

# 3. Online Bayesian Change-Point Probability Proxy
def compute_changepoint_prob(data):
    cp_probs = np.zeros(len(data))
    for i in range(1, len(data)):
        diff = abs(data[i] - data[i-1])
        cp_probs[i] = 1 - np.exp(-diff)
    return cp_probs

df_netf['cp_prob'] = compute_changepoint_prob(df_netf[target_col].values)

# NETF Composite Transition Fingerprint Scoring Model
features_netf = ['ar1_autocorr', 'rolling_variance', 'cross_correlation_mean', 'cp_prob']
X_netf = df_netf[features_netf]
y_netf = (df_netf['phase_state'] >= 1).astype(int)

logistic = LogisticRegression()
logistic.fit(X_netf, y_netf)
df_netf['transition_score'] = logistic.predict_proba(X_netf)[:, 1]

print("NETF Transition Fingerprint Computed Successfully.")
print(df_netf[['Date', 'ar1_autocorr', 'rolling_variance', 'cp_prob', 'transition_score']].tail())


## Stage 3 — Irreversibility-Constrained Causal Twin (ICCT)


In [ ]:
class ICCT_CausalDigitalTwin:
    def __init__(self, feature_names):
        self.feature_names = feature_names
        self.models = {}
        self.persistent_damage = np.zeros(len(feature_names))
        
    def fit_dependency_graph(self, X_df):
        for col in self.feature_names:
            y = X_df[col]
            X_lagged = X_df.drop(columns=[col]).shift(1).fillna(0)
            lasso = Lasso(alpha=0.05)
            lasso.fit(X_lagged, y)
            self.models[col] = lasso
            
    def predict_next_state(self, current_state_dict):
        next_state = {}
        input_array = np.array([current_state_dict[f] for f in self.feature_names]).reshape(1, -1)
        for col in self.feature_names:
            next_state[col] = float(self.models[col].predict(input_array)[0])
        return next_state

    def counterfactual_simulation(self, current_state, intervention_var, intervention_val, steps=6):
        trajectory = []
        state = current_state.copy()
        
        state[intervention_var] = intervention_val
        
        accumulated_damage = 0.0
        for t in range(steps):
            state = self.predict_next_state(state)
            if intervention_var in state:
                state[intervention_var] = intervention_val
            
            damage_t = max(0, state['NYFed_Financial_Stress_scaled'] - 0.5) * 0.2
            accumulated_damage += damage_t
            
            trajectory.append((state.copy(), accumulated_damage))
            
        return trajectory

# Initialize and fit ICCT
twin = ICCT_CausalDigitalTwin(scaled_cols)
twin.fit_dependency_graph(df_state[scaled_cols])

# Run counterfactual example
latest_state = df_state[scaled_cols].iloc[-1].to_dict()
cf_result = twin.counterfactual_simulation(latest_state, 'FRED_Credit_Cond_scaled', intervention_val=1.5, steps=6)

print(f"Counterfactual simulation completed. Final estimated accumulated damage: {cf_result[-1][1]:.4f}")


## Stage 4 — Reversibility-Budgeted Crisis Controller (RBCC)


In [ ]:
class RBCC_Controller:
    def __init__(self, digital_twin, alpha=1.0, beta=1.5, gamma=0.5, delta=0.8):
        self.twin = digital_twin
        self.alpha = alpha  # Weight for Systemic Risk Reduction
        self.beta = beta    # Weight for Irreversibility Cost
        self.gamma = gamma  # Weight for Intervention Cost
        self.delta = delta  # Weight for Counterfactual Uncertainty
        
    def evaluate_interventions(self, current_state, candidate_interventions):
        rankings = []
        
        base_traj = self.twin.counterfactual_simulation(current_state, intervention_var=list(candidate_interventions.keys())[0], intervention_val=current_state[list(candidate_interventions.keys())[0]], steps=6)
        base_risk = base_traj[-1][0]['NYFed_Financial_Stress_scaled']
        
        for var, val in candidate_interventions.items():
            traj = self.twin.counterfactual_simulation(current_state, intervention_var=var, intervention_val=val, steps=6)
            
            final_risk = traj[-1][0]['NYFed_Financial_Stress_scaled']
            risk_reduction = base_risk - final_risk
            irreversibility_cost = traj[-1][1]
            intervention_cost = abs(val - current_state[var]) * 0.5
            uncertainty = np.random.uniform(0.05, 0.2)
            
            rbcc_score = (self.alpha * risk_reduction) - (self.beta * irreversibility_cost) - (self.gamma * intervention_cost) - (self.delta * uncertainty)
            
            rankings.append({
                'Intervention_Variable': var,
                'Intervention_Value': val,
                'Risk_Reduction': risk_reduction,
                'Irreversibility_Cost': irreversibility_cost,
                'Intervention_Cost': intervention_cost,
                'Uncertainty': uncertainty,
                'RBCC_Score': rbcc_score
            })
            
        df_rankings = pd.DataFrame(rankings).sort_values(by='RBCC_Score', ascending=False)
        return df_rankings

controller = RBCC_Controller(twin)
candidate_actions = {
    'FRED_Credit_Cond_scaled': 1.2,
    'ECB_Bank_BalanceSheet_scaled': 0.8,
    'FRED_Yield_Spread_scaled': -0.5
}

recommendations = controller.evaluate_interventions(latest_state, candidate_actions)
print("--- RBCC Ranked Intervention Recommendations ---")
print(recommendations)


## Stage 5 — Integrated Validation, Robustness & Computational Profiling


In [ ]:
# 1. Walk-Forward Validation Protocol
print("Executing Walk-Forward Validation...")
train_sizes = range(100, len(df_state) - 12, 12)
aucs = []

for t_size in train_sizes:
    train_df = df_netf.iloc[:t_size]
    test_df = df_netf.iloc[t_size:t_size+12]
    
    model = LogisticRegression()
    model.fit(train_df[features_netf], (train_df['phase_state'] >= 1).astype(int))
    
    preds = model.predict_proba(test_df[features_netf])[:, 1]
    true_labels = (test_df['phase_state'] >= 1).astype(int)
    
    if len(np.unique(true_labels)) > 1:
        auc = roc_auc_score(true_labels, preds)
        aucs.append(auc)

print(f"Mean Walk-Forward ROC-AUC: {np.mean(aucs):.4f}")

# 2. Runtime and Memory Profiling
process = psutil.Process(os.getpid())
start_mem = process.memory_info().rss / (1024 * 1024)
start_time = time.time()

# Benchmark full pipeline execution
_ = twin.counterfactual_simulation(latest_state, 'FRED_Credit_Cond_scaled', 1.0, steps=12)
_ = controller.evaluate_interventions(latest_state, candidate_actions)

end_time = time.time()
end_mem = process.memory_info().rss / (1024 * 1024)

print("\n--- Computational Profiling Report ---")
print(f"Inference Latency: {(end_time - start_time)*1000:.2f} ms")
print(f"Peak Memory Consumption: {end_mem - start_mem:.2f} MB")
print(f"Total Process RAM Usage: {end_mem:.2f} MB")
